# Bond P&L Attribution

## Overview

This notebook demonstrates a typical lifecycle for a GBP bond held in a EUR portfolio, how the P&L from this can be attributed, and how it can be reported in another currency (USD).  The lifecycle includes the following steps:

1. **Setup Transaction** — Purchase 100 units of a bond at 0.95 clean price (2024-01-01)
2. **Coupon Instrument Event** — Generate a semi-annual coupon payment of 0.10/unit (2024-06-30)
3. **Sale Transaction** — Sell 25 units at 0.99 clean price (2024-11-20)
4. **Maturity Instrument Event** — Redeem remaining 75 units at par (2024-12-31)

## Standard LUSID P&L breakdown over period 2024-10-19 to 2025-01-02

| Bucket | Component | Amount | Description |
|---|---|---|---|
| CapGain | Change in Unrealised | -3.00 | 100 units purchased @ .95, price at start of period .98, unrealised at end 0 |
| CapGain | Realised (Sale) | 1.00 | 25 units purchased @ .95, sold @ .99 -> 0.04 * 25 | 
| CapGain | Realised (Maturity) | 3.75 | 75 units purchased at .95 -> 0.05 * 75 |
| Carry | Carry — Sold Interest | 2.25 | 25 units, unit accrual of 0.09 |
| Carry | Interest | 7.50 | 75 units, unit accrual of 0.1 |
| Carry | Delta Accrued | -8.00 | 100 units, unit accrual of 0.8 at start of period, zero at end |
| **TOTAL** | **3.50** |

## Split between Held and Trading P&L over the same period

| Attribution | Component | Amount | Description |
|---|---|---|---|
| Holding | Market | -3.00 | 100 units purchased @ .95, price at start of period .98, unrealised at end of 0 |
| Holding | Realised | 5 | 100 units purchased @ .95, matured at par -> 0.05 * 100 |
| Holding | Interest | 10 | 100 units * 0.1 coupon rate |
| Holding | Delta Accrued | -8 | Accrual 19/10 0.8, accrual at end 0 |
| **Holding Total** | | 4 |
| Trading | Market | -0.25 | 25 units sold @ .99, matured at 1 -> 0.01 * -25 |
| Trading | Interest | -0.25 | -25 units @ coupon rate 0.01 |
| **Trading Total** | | -0.5 |
| **TOTAL** | **3.50** |

**Please note**: The A2B endpoint which splits holding and trading returns ([link](https://www.lusid.com/docs/api/lusid/endpoints/transaction-portfolios/GetA2BMovementsTradingVsHolding)) is currently an experimental endpoint so requires a special licence.  Please contact support@lusid.com to request the licence `experimental-api-transactionportfolios-geta2bmovementstradingvsholding` for your domain.


In [ ]:
# Import common libraries
import json
import datetime as dt
import io
import os
import pandas as pd
import logging
import pytz
import random
from datetime import datetime, timezone
from IPython.core.display import HTML
from collections import defaultdict
logging.basicConfig(level = logging.ERROR)

# Import LUSID libraries
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils import cocoon as cocoon

import fbnconfig

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)

# Confirm success
api_url = api_factory.get_api_configuration().host.replace("api","")

print ('LUSID Environment :', api_url + "docs")
display(pd.DataFrame(api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().to_dict()))

In [ ]:
property_definitions_api       = api_factory.build(lu.PropertyDefinitionsApi)
instruments_api                = api_factory.build(lu.InstrumentsApi)
transaction_portfolios_api     = api_factory.build(lu.TransactionPortfoliosApi)
derived_portfolios_api         = api_factory.build(lu.DerivedTransactionPortfoliosApi)
portfolios_api                 = api_factory.build(lu.PortfoliosApi)
corporate_action_sources_api   = api_factory.build(lu.CorporateActionSourcesApi)
instrument_events_api          = api_factory.build(lu.InstrumentEventsApi)
configuration_recipe_api       = api_factory.build(lu.ConfigurationRecipeApi)
quotes_api                     = api_factory.build(lu.QuotesApi)
aggregation_api                = api_factory.build(lu.AggregationApi)

print("APIs initialized")

---
## Section 1: Configuration

Define scope, key dates, and bond parameters for the scenario.

In [ ]:
# ── Scope & identifiers ───────────────────────────────────────────────────────
scope = "holding-vs-trading-pnl"
transaction_type_scope = "FINBOURNE_RECOMMENDED_TXN_TYPES"
portfolio_code = "BOND-LIFECYCLE"
cas_code = "BOND-LIFECYCLE-CAS"

bond_client_internal = "BOND-001"
bond_isin            = "GB00FR4279001X"
bond_currency        = "GBP"
portfolio_currency   = "EUR"
report_currency      = "USD"
bond_principal       = 1.0

recipe_code = "BOND-LIFECYCLE-RECIPE"

# ── Coupon parameters ─────────────────────────────────────────────────────────
bond_coupon_rate = 0.10

# ── Transaction parameters ────────────────────────────────────────────────────
buy_units              = 100
buy_clean_price        = 0.95    # unit cost / book cost per unit

sell_units  = 25
sell_price  = 0.99               # clean trade price at sale

maturity_price = 1.0             # par redemption

# ── Key dates ─────────────────────────────────────────────────────────────────
start_date           = dt.datetime(2024,  1,  1, tzinfo=pytz.utc)
buy_date             = dt.datetime(2024,  1,  1, tzinfo=pytz.utc)
buy_settlement_date  = dt.datetime(2024,  1,  1, tzinfo=pytz.utc)
sale_date            = dt.datetime(2024,  11, 20, tzinfo=pytz.utc)
sale_settlement_date = dt.datetime(2024, 11,  24, tzinfo=pytz.utc)
maturity_date        = dt.datetime(2024, 12, 31, tzinfo=pytz.utc)

report_period_start_date = dt.datetime(2024,  10,  19, tzinfo=pytz.utc)
report_period_end_date   = dt.datetime(2025, 1, 2, tzinfo=pytz.utc)

shk_property_domain = "Transaction"
shk_property_code = "SHK"
shk_property_key = f"{shk_property_domain}/{scope}/{shk_property_code}"
shk_property_value = "BondScenario"

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Scope:     {scope}")
print(f"Portfolio: {portfolio_code}")
print(f"Bond:      {bond_client_internal} ({bond_isin})")

In [ ]:
with open(f"../../setup/{transaction_type_scope}.json") as f:
    deployment = fbnconfig.undump_deployment(json.load(f))

actions = fbnconfig.deploy(
    deployment, 
    api_factory.get_api_configuration().host,
    api_factory.get_api_configuration().access_token,
    write_control=0,  # 1=upsert, 2=no-update, 4=no-delete, 8=delete-if-owned
)
actions

In [ ]:
configuration_recipe = lm.ConfigurationRecipe(
    scope=scope,
    code=recipe_code,
    market=lm.MarketContext(
        market_rules=[
            lm.MarketDataKeyRule(
                key="Quote.ClientInternal.*",
                supplier="Lusid",
                data_scope=scope,
                quote_type="Price",
                field="mid",
                quote_interval="10Y.0D"
            ),
            lm.MarketDataKeyRule(
                key="Fx.CurrencyPair.*",
                supplier="Lusid",
                data_scope=scope,
                quote_type="Rate",
                field="mid",
                quote_interval="10Y.0D"
            ),
        ],
        options=lm.MarketOptions(
            default_supplier="Lusid",
            default_instrument_code_type="ClientInternal",
            default_scope=scope,
            attempt_to_infer_missing_fx=True
        ),
    ),
    pricing=lm.PricingContext(
        options={"AllowPartiallySuccessfulEvaluation": True},
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="BondLookupPricer",
                instrument_type="Bond",
                parameters="{}",
            )
        ]
    ),
)

configuration_recipe_api.upsert_configuration_recipe(
    upsert_recipe_request=lm.UpsertRecipeRequest(
        configuration_recipe=configuration_recipe
    )
)
print(f"Upserted recipe: {scope}/{recipe_code}")

---
## Section 1b: Quotes

Upsert market prices for the bond into the quotes store.
These are used by the recipe for valuation (A2B, holdings with market value, etc.).

| Date | Price | Notes |
|---|---|---|
| 2024-10-19 | 0.98 | Market price at start of period |

In [ ]:
quotes = {
    "bond-price-start": lm.UpsertQuoteRequest(
        quote_id=lm.QuoteId(
            quote_series_id=lm.QuoteSeriesId(
                provider="Lusid",
                instrument_id=bond_client_internal,
                instrument_id_type="ClientInternal",
                quote_type="Price",
                field="mid"
            ),
            effective_at=report_period_start_date.isoformat()
        ),
        metric_value=lm.MetricValue(value=0.98, unit=bond_currency)
    )
}

try:
    response = quotes_api.upsert_quotes(scope=scope, request_body=quotes)
    print(f"Upserted {len(response.values)} quote(s)")
    for key, q in response.values.items():
        print(f"   {q.quote_id.effective_at}  {q.quote_id.quote_series_id.instrument_id}  mid = {q.metric_value.value} {q.metric_value.unit}")
    if response.failed:
        print(f"Failed: {response.failed}")
except ApiException as e:
    error = json.loads(e.body)
    print(f"Error: {error.get('title', 'Unknown error')}")

---
## Section 1c: FX Rates

Upsert FX rates into the quotes store. The bond is in GBP and the portfolio in EUR, so the bond is converted into the portfolio currency using EUR/GBP, and movements in that rate over the bond's life produce FX P&L. Rates are quoted in the market-standard direction; LUSID inverts them where it needs the opposite, here GBP to EUR.

GBP/USD and EUR/USD are only needed in Section 9b, where the P&L is reported in USD: the bond's GBP P&L is converted at GBP/USD, and the portfolio's EUR FX P&L at EUR/USD.

Where all three rates are given for a date, they are consistent: EUR/USD equals EUR/GBP × GBP/USD (e.g. 0.83 × 1.30 = 1.079 at the valuation date).

| Date | EUR/GBP | GBP/USD | EUR/USD | Notes |
|---|---|---|---|---|
| 2024-01-01 | 0.86 | | | Buy |
| 2024-10-19 | 0.84 | 1.25 | 1.05 | Start of period |
| 2024-11-20 | 0.83 | | | Sale |
| 2024-12-31 | 0.82 | | | Maturity |
| 2025-01-02 | 0.83 | 1.30 | 1.079 | Valuation date |

In [ ]:
portfolio_bond_pair = f"{portfolio_currency}/{bond_currency}"
bond_report_pair = f"{bond_currency}/{report_currency}"
portfolio_report_pair = f"{portfolio_currency}/{report_currency}"

fx_rates = {
    portfolio_bond_pair: {
        buy_date:                 0.86,
        report_period_start_date: 0.84,
        sale_date:                0.83,
        maturity_date:            0.82,
        report_period_end_date:   0.83,
    },
    bond_report_pair: {
        report_period_start_date: 1.25,
        report_period_end_date:   1.30,
    },
}
fx_rates[portfolio_report_pair] = {
    effective_at: round(fx_rates[portfolio_bond_pair][effective_at] * rate, 6)
    for effective_at, rate in fx_rates[bond_report_pair].items()
}

fx_quotes = {
    f"{pair.replace('/', '')}-{effective_at:%Y%m%d}": lm.UpsertQuoteRequest(
        quote_id=lm.QuoteId(
            quote_series_id=lm.QuoteSeriesId(
                provider="Lusid",
                instrument_id=pair,
                instrument_id_type="CurrencyPair",
                quote_type="Rate",
                field="mid"
            ),
            effective_at=effective_at.isoformat()
        ),
        metric_value=lm.MetricValue(value=rate, unit=pair)
    )
    for pair, rates in fx_rates.items()
    for effective_at, rate in rates.items()
}

try:
    response = quotes_api.upsert_quotes(scope=scope, request_body=fx_quotes)
    print(f"Upserted {len(response.values)} FX quote(s)")
    for q in sorted(response.values.values(), key=lambda q: (q.quote_id.quote_series_id.instrument_id, q.quote_id.effective_at)):
        print(f"   {q.quote_id.effective_at}  {q.quote_id.quote_series_id.instrument_id}  mid = {q.metric_value.value}")
    if response.failed:
        print(f"Failed: {response.failed}")
except ApiException as e:
    error = json.loads(e.body)
    print(f"Error: {error.get('title', 'Unknown error')}")

---
## Section 2: Bond Instrument

Create the bond instrument in LUSID with inline flow conventions.
All economic details (coupon rate, maturity, principal) are captured in the instrument definition.

In [ ]:
# Inline flow conventions — no pre-existing convention needed
flow_conventions = lm.FlowConventions(
    currency=bond_currency,
    payment_frequency="12M",
    day_count_convention="Actual365",
    roll_convention="MF",
    payment_calendars=[],
    reset_calendars=[]
)

bond_definition = lm.Bond(
    instrument_type="Bond",
    start_date=start_date,
    maturity_date=maturity_date,
    dom_ccy=bond_currency,
    principal=bond_principal,
    coupon_rate=bond_coupon_rate,
    flow_conventions=flow_conventions,
    identifiers={}
)

bond_instrument = lm.InstrumentDefinition(
    name="Attribution Demo Bond",
    identifiers={
        "ClientInternal": lm.InstrumentIdValue(value=bond_client_internal),
        "Isin":           lm.InstrumentIdValue(value=bond_isin)
    },
    definition=bond_definition
)

try:
    response = instruments_api.upsert_instruments(
        scope="default",
        request_body={"bond": bond_instrument}
    )
    luid = list(response.values.values())[0].lusid_instrument_id
    print(f"Upserted bond: Demo Bond")
    print(f"   ClientInternal : {bond_client_internal}")
    print(f"   ISIN           : {bond_isin}")
    print(f"   LUID           : {luid}")
    # print(f"   Coupon rate    : {bond_coupon_rate*100:.0f}% p.a., {coupon_per_unit}/unit per semi-annual period")
    print(f"   Maturity       : {maturity_date.date()} @ par ({maturity_price})")
except ApiException as e:
    error = json.loads(e.body)
    print(f"Error: {error.get('title', 'Unknown error')}")

---
## Section 3: Corporate Action Source

Create a transaction portfolio and a description property to label this as the scenario.

In [ ]:
# Create portfolio description property
try:
    prop_def = lm.CreatePropertyDefinitionRequest(
        domain=shk_property_domain,
        scope=scope,
        code=shk_property_code,
        display_name="SHK",
        data_type_id=lm.ResourceId(scope="system", code="string"),
        life_time="Perpetual",
        constraint_style="Property"
    )
    response = property_definitions_api.create_property_definition(
        create_property_definition_request=prop_def
    )
    print(f"Created property: {response.key}")
except ApiException as e:
    error = json.loads(e.body)
    if error.get("name") == "PropertyAlreadyExists":
        print(f"Property already exists: {shk_property_code}")
    else:
        print(f"Error: {error.get('title', 'Unknown error')}")

In [ ]:
try:
    corporate_action_sources_api.delete_corporate_action_source(scope=scope, code=cas_code)
    print(f"Deleted existing CAS: {scope}/{cas_code}")
except ApiException as e:
    error = json.loads(e.body)
    if error.get("name") == "NotFound":
        print(f"No existing CAS to delete: {scope}/{cas_code}")
    else:
        print(f"Error: {error.get('title', 'Unknown error')}")

In [ ]:
# Create corporate action source — required to process instrument events against the portfolio
try:
    response = corporate_action_sources_api.create_corporate_action_source(
        create_corporate_action_source_request=lm.CreateCorporateActionSourceRequest(
            scope=scope,
            code=cas_code,
            display_name="Bond Lifcycle Corporate Action Source",
            description="Corporate action source for the bond lifecycle scenario"
        )
    )
    print(f"Created corporate action source: {response.id.scope}/{response.id.code}")

    instrument_events = corporate_action_sources_api.get_instrument_events(scope=scope, code=cas_code).values
    if len(instrument_events) > 0:
        raise Exception(f"Expected no events for new CAS, but found {len(instrument_events)}")
    
except ApiException as e:
    error = json.loads(e.body)
    if error.get("name") == "AlreadyExists":
        print(f"Corporate action source already exists: {scope}/{cas_code}")
    else:
        print(f"Error: {error.get('title', 'Unknown error')}")

## Portfolio

In [ ]:
try:
    portfolios_api.delete_portfolio(scope=scope, code=portfolio_code)
    print(f"Deleted existing portfolio: {scope}/{portfolio_code}")
except ApiException as e:
    error = json.loads(e.body)
    if error.get("name") == "PortfolioNotFound":
        print(f"Portfolio does not exist, nothing to delete")
    else:
        print(f"Error: {error.get('title', 'Unknown error')}")

In [ ]:
response = transaction_portfolios_api.create_portfolio(
    scope=scope,
    create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name="Bond Lifecycle Demo",
        code=portfolio_code,
        base_currency=portfolio_currency,
        created=buy_date,
        transaction_type_scope=transaction_type_scope,
        corporate_action_source_id=lm.ResourceId(scope=scope, code=cas_code),
        instrumentEventConfiguration=lm.InstrumentEventConfiguration(
            recipeId=lm.ResourceId(scope=scope, code=recipe_code)
        ),
    )
)
print(f"Created portfolio: {response.id.scope}/{response.id.code}")
print(f"   Base currency          : {portfolio_currency}")
print(f"   Created                : {buy_date.date()}")
print(f"   Corporate action source: {scope}/{cas_code}")

---
## Section 4: Setup Transaction — Buy 100 Units

Purchase 100 units of the bond at the start of the coupon period.

| | Value |
|---|---|
| Units | 100 |
| Clean price | 0.95 |
| Total consideration | GBP 95.00 |
| Unit cost (book cost/unit) | 0.95 |
| Accrued at purchase | 0 (start of coupon period) |

In [ ]:
transaction_id = "TXN-BUY-001"
buy_transaction = lm.TransactionRequest(
    transaction_id=transaction_id,
    type="Buy",
    instrument_identifiers={
        "Instrument/default/ClientInternal": bond_client_internal
    },
    transaction_date=buy_date.isoformat(),
    settlement_date=buy_settlement_date.isoformat(),
    units=buy_units,
    transaction_price=lm.TransactionPrice(
        price=buy_clean_price,
        type="CleanPrice"
    ),
    total_consideration=lm.CurrencyAndAmount(
        amount=0,
        currency=bond_currency
    ),
    properties={
        shk_property_key: lm.PerpetualProperty(
           key = shk_property_key,
           value = lm.PropertyValue(label_value=shk_property_value)
        )
    },
    source="default"
)

response = transaction_portfolios_api.upsert_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_request=[buy_transaction]
)
print(f"Upserted buy transaction: {transaction_id}")
print(f"   Units          : +{buy_units}")
print(f"   Clean price    : {buy_clean_price}")
print(f"   Trade date     : {buy_date.date()}")
print(f"   Settlement     : {buy_settlement_date.date()}")

---
## Section 6: Sale Transaction — Sell 25 Units

Sell 25 of the 100 units at a clean price of 0.99, leaving a residual position of 75 units.

| | Value |
|---|---|
| Units sold | -25 |
| Clean trade price | 0.99 |
| Clean consideration | GBP 24.75 |
| Book cost removed (25 × 0.95) | GBP 23.75 |
| Realised gain (clean) | GBP 1.00 |
| Residual position | 75 units |

In [ ]:
transaction_id = "TXN-SELL-001"

sell_transaction = lm.TransactionRequest(
    transaction_id=transaction_id,
    type="Sell",
    instrument_identifiers={
        "Instrument/default/ClientInternal": bond_client_internal
    },
    transaction_date=sale_date.isoformat(),
    settlement_date=sale_settlement_date.isoformat(),
    units=sell_units,
    transaction_price=lm.TransactionPrice(
        price=sell_price,
        type="CleanPrice"
    ),
    total_consideration=lm.CurrencyAndAmount(
        amount=0,
        currency=bond_currency
    ),
    properties={
        shk_property_key: lm.PerpetualProperty(
           key = shk_property_key,
           value = lm.PropertyValue(label_value=shk_property_value)
        )
    },
    source="default"
)

response = transaction_portfolios_api.upsert_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_request=[sell_transaction]
)

print(f"Upserted sell transaction: {transaction_id}")
print(f"   Units            : -{sell_units}")
print(f"   Clean price      : {sell_price}")
print(f"   Trade date       : {sale_date.date()}")
print(f"   Settlement       : {sale_settlement_date.date()}")

---
## Section 8: Verification

In [ ]:
# Retrieve Transactions

transactions = transaction_portfolios_api.build_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_query_parameters=lm.TransactionQueryParameters(
        startDate=start_date.isoformat(),
        endDate=report_period_end_date.isoformat())
    )

pd.DataFrame(transactions.values)

In [ ]:
# Valuations at each stage of the lifecycle
checkpoints = {
    "Period start":  report_period_start_date,
    "After sale":    sale_date,
    "After maturity": maturity_date,
    "Period end":    report_period_end_date,
}

for label, check_date in checkpoints.items():
    print(f"\n{label}: {check_date.strftime('%Y-%m-%d')}")

    valuation_request = lm.ValuationRequest(
        recipe_id=lm.ResourceId(scope=scope, code=recipe_code),
        metrics=[
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Sum"),
            lm.AggregateSpec(key="Valuation/PV", op="Sum"),
            lm.AggregateSpec(key="Valuation/Accrued", op="Sum"),
            lm.AggregateSpec(key="Holding/default/Cost", op="Sum"),
        ],
        group_by=["Instrument/default/LusidInstrumentId"],
        valuation_schedule=lm.ValuationSchedule(effective_at=check_date.isoformat()),
        portfolio_entity_ids=[lm.PortfolioEntityId(
            scope=scope,
            code=portfolio_code,
            portfolio_entity_type="SinglePortfolio"
        )]
    )

    valuation = aggregation_api.get_valuation(valuation_request=valuation_request)
    if valuation.data:
        df = pd.DataFrame(valuation.data).rename(columns={
            "Value(Instrument/default/LusidInstrumentId)": "Instrument",
            "Sum(Holding/default/Units)":                  "Units",
            "Sum(Valuation/PV)":                           "PV (GBP)",
            "Sum(Holding/default/Cost)":                   "Cost (GBP)",
        })
        display(df)
    else:
        print("   No valuation data")


---
## Section 9a: Calculation of P&L

Use the A2B report to calculate P&L for the period

In [ ]:
# A2B P&L across the full lifecycle
a2b_response = transaction_portfolios_api.get_a2_b_movements_trading_vs_holding(
    scope=scope,
    code=portfolio_code,
    from_effective_at=report_period_start_date.isoformat(),
    to_effective_at=report_period_end_date.isoformat(),
    recipe_id_scope=scope,
    recipe_id_code=recipe_code
)

if a2b_response.values:
    # Aggregate records by group_id
    groups = defaultdict(list)
    for r in a2b_response.values:
        groups[r.group_id].append(r)

    for group_id, records in groups.items():
        # Extract common fields from first record
        first = records[0]
        print(f"{'='*80}")
        print(f"Group: {group_id}")
        print(f"  Instrument: {first.instrument_uid}  Holding type: {first.holding_type}  Currency: {first.currency}")
        print(f"  Records: {len(records)}")
        print()

        # Summarise movements (flows, carry, gains)
        total_flows = 0.0
        total_carry = 0.0
        total_gains = 0.0

        for r in records:
            # if r.movement_name is None:
            #     continue
            if r.start and r.start.holding_currency:
                hc = r.start.holding_currency
                print(f"  START  units={r.units:,.2f}  total={hc.total:,.2f}  components={dict(hc.components)}")

            parts = []
            if r.flows and r.flows.holding_currency:
                fc = r.flows.holding_currency
                total_flows += fc.total
                parts.append(f"flows={fc.total:,.2f} {dict(fc.components)}")
            if r.carry and r.carry.holding_currency:
                cc = r.carry.holding_currency
                total_carry += cc.total
                parts.append(f"carry={cc.total:,.2f} {dict(cc.components)}")
            if r.gains and r.gains.holding_currency:
                gc = r.gains.holding_currency
                total_gains += gc.total
                parts.append(f"gains={gc.total:,.2f} {dict(gc.components)}")

            date_str = r.effective_date.strftime("%Y-%m-%d") if r.effective_date else "N/A"
            units_str = f"units={r.units:,.2f}" if r.units else ""

            if r.movement_name is not None:
                print(f"  MOVE   {date_str}  {r.movement_name:<50s} {units_str}  {', '.join(parts)}")

            if r.end and r.end.holding_currency:
                hc = r.end.holding_currency
                print(f"  END    units={r.units:,.2f}  total={hc.total:,.2f}  components={dict(hc.components)}")

        print()
        print(f"  TOTALS  flows={total_flows:,.2f}  carry={total_carry:,.2f}  gains={total_gains:,.2f}  net={total_flows + total_carry + total_gains:,.2f}")
        print()
else:
    print("No A2B records returned — check date range and that instrument events have been applied")

Use `ProfitAndLoss/Attribution`AddressKeys to retrieve the same data

In [ ]:
# Query the list of addresskeys from getqueryablekeys
response = aggregation_api.get_queryable_keys(filter="addresskey sw 'ProfitAndLoss/Attribution'")

address_keys_df = pd.DataFrame(
    [{"AddressKey": k.address_key, "Description": k.description} for k in response.values]
).sort_values("AddressKey").reset_index(drop=True)

display(address_keys_df)

In [ ]:

# Valuations using new addresskeys
valuation_request = lm.ValuationRequest(
    recipe_id=lm.ResourceId(scope=scope, code=recipe_code),
    metrics=[
        lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
        lm.AggregateSpec(key="Holding/default/Units", op="Sum"),
        lm.AggregateSpec(key="Valuation/PV", op="Sum"),
        lm.AggregateSpec(key="Holding/default/Cost", op="Sum"),
        lm.AggregateSpec(key="ProfitAndLoss/Attribution/Total/Held", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": "2024-10-19T23:59:59.999999Z"
                         }, 
                         op="Value"),
        lm.AggregateSpec(key="ProfitAndLoss/Attribution/Held/Market", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": "2024-10-19T23:59:59.999999Z"
                         }, 
                         op="Value"),
        lm.AggregateSpec(key="ProfitAndLoss/Attribution/Held/Other", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": "2024-10-19T23:59:59.999999Z"
                         }, 
                         op="Value"),
        lm.AggregateSpec(key="ProfitAndLoss/Attribution/Total/Trading", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": "2024-10-19T23:59:59.999999Z"
                         }, 
                         op="Value"),
        lm.AggregateSpec(key="ProfitAndLoss/Attribution/Trading/Market", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": "2024-10-19T23:59:59.999999Z"
                         }, 
                         op="Value"),
        lm.AggregateSpec(key="ProfitAndLoss/Attribution/Trading/Other", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": "2024-10-19T23:59:59.999999Z"
                         }, 
                         op="Value"),
         # Standard ProfitAndLoss addresskey to trigger ProfitAndLossTask to no-longer-held positions
         lm.AggregateSpec(key="ProfitAndLoss/Total", 
                         options={
                            "Window" : "YTD",
                            "StartDateOverrideUTC": report_period_start_date.isoformat()
                         }, 
                         op="Value"),

    ],
    group_by=["Instrument/default/LusidInstrumentId"],
    valuation_schedule=lm.ValuationSchedule(effective_at=report_period_end_date.isoformat()),
    portfolio_entity_ids=[lm.PortfolioEntityId(
        scope=scope,
        code=portfolio_code,
        portfolio_entity_type="SinglePortfolio"
    )]
)

valuation = aggregation_api.get_valuation(valuation_request=valuation_request)
if valuation.data:
    df = pd.DataFrame(valuation.data).rename(columns={
        "Value(Instrument/default/LusidInstrumentId)": "Instrument",
        "Sum(Holding/default/Units)":                  "Units",
        "Sum(Valuation/PV)":                           "PV (GBP)",
        "Sum(Holding/default/Cost)":                   "Cost (GBP)",
        "Sum(ProfitAndLoss/Attribution/Total/Held)":  "P&L Attribution (YTD)",
    })
    df.columns = [column.split("(Window=")[0] for column in df.columns]
    print(f"P&L window: {report_period_start_date:%Y-%m-%d} to {report_period_end_date:%Y-%m-%d}")
    display(df)
else:
    print("   No valuation data")





---
## Section 9b: P&L in Reporting Currency

The `ProfitAndLoss/.../ReportCcy` AddressKeys return the windowed P&L converted into the valuation request's `report_currency`. The conversion is applied once, to the window's P&L, at the FX rate on the valuation's effectiveAt - not row by row at each date's own rate.

The bond is GBP, the portfolio EUR, and the P&L is reported in USD using the rates upserted in Section 1c. Each family of report currency keys converts a different value, at the valuation date's rate:

| AddressKeys | Converts | Rate |
|---|---|---|
| `Market`, `Other` and `Total` | Domestic (GBP) P&L | GBP/USD |
| `Fx` | Portfolio (EUR) FX P&L | EUR/USD |

The ReportCcy keys are a spot conversion at the valuation date, so they differ from the PortfolioCcy keys even when the reporting currency is the portfolio currency.

`Total/ReportCcy` excludes FX P&L. It converts the domestic `Total`, which has no FX component; the FX P&L is reported separately in `Total/Fx/ReportCcy`.

LUSID returns these values unrounded unless the recipe sets `applyIso4217Rounding`. Here they are shown rounded to 4 decimal places where they have more.

In [ ]:
pnl_options = {
    "Window": "YTD",
    "StartDateOverrideUTC": report_period_start_date.isoformat()
}

report_ccy_to_base_key = {
    "ProfitAndLoss/Realised/Market/ReportCcy":   ("ProfitAndLoss/Realised/Market",            bond_currency),
    "ProfitAndLoss/Unrealised/Market/ReportCcy": ("ProfitAndLoss/Unrealised/Market",          bond_currency),
    "ProfitAndLoss/Total/Market/ReportCcy":      ("ProfitAndLoss/Total/Market",               bond_currency),
    "ProfitAndLoss/Total/Other/ReportCcy":       ("ProfitAndLoss/Total/Other",                bond_currency),
    "ProfitAndLoss/Total/ReportCcy":             ("ProfitAndLoss/Total",                      bond_currency),
    "ProfitAndLoss/Realised/Fx/ReportCcy":       ("ProfitAndLoss/Realised/Fx/PortfolioCcy",   portfolio_currency),
    "ProfitAndLoss/Unrealised/Fx/ReportCcy":     ("ProfitAndLoss/Unrealised/Fx/PortfolioCcy", portfolio_currency),
    "ProfitAndLoss/Total/Fx/ReportCcy":          ("ProfitAndLoss/Total/Fx/PortfolioCcy",      portfolio_currency),
}
rate_to_report_ccy = {
    bond_currency:      fx_rates[bond_report_pair][report_period_end_date],
    portfolio_currency: fx_rates[portfolio_report_pair][report_period_end_date],
}

pnl_keys = list(report_ccy_to_base_key) + [base_key for base_key, _ in report_ccy_to_base_key.values()]

valuation = aggregation_api.get_valuation(valuation_request=lm.ValuationRequest(
    recipe_id=lm.ResourceId(scope=scope, code=recipe_code),
    metrics=[lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value")]
          + [lm.AggregateSpec(key=key, options=pnl_options, op="Sum") for key in pnl_keys],
    group_by=["Instrument/default/LusidInstrumentId"],
    report_currency=report_currency,
    valuation_schedule=lm.ValuationSchedule(effective_at=report_period_end_date.isoformat()),
    portfolio_entity_ids=[lm.PortfolioEntityId(
        scope=scope,
        code=portfolio_code,
        portfolio_entity_type="SinglePortfolio"
    )]
))

def pnl_value(row, key):
    return next(value for column, value in row.items() if column.startswith(f"Sum({key}("))

def up_to_4dp(value):
    whole, fraction = f"{value:,.4f}".split(".")
    return f"{whole}.{fraction.rstrip('0').ljust(2, '0')}"

def with_currency(value, currency):
    return f"{currency} {up_to_4dp(value)}"

for row in valuation.data:
    print(f"\n{row['Instrument/default/LusidInstrumentId']}")
    df = pd.DataFrame([
        {
            "AddressKey":                                          report_ccy_key,
            "Converts":                                            base_key,
            "P&L":                                                 with_currency(pnl_value(row, base_key), from_currency),
            "Rate":                                                f"{from_currency}/{report_currency} {rate_to_report_ccy[from_currency]:.3f}",
            f"Expected P&L in Report Currency ({report_currency})": up_to_4dp(pnl_value(row, base_key) * rate_to_report_ccy[from_currency]),
            f"P&L in Report Currency ({report_currency})":          up_to_4dp(pnl_value(row, report_ccy_key)),
        }
        for report_ccy_key, (base_key, from_currency) in report_ccy_to_base_key.items()
    ])
    display(df)

    fx_fmt = with_currency(pnl_value(row, "ProfitAndLoss/Total/Fx/ReportCcy"), report_currency)
    total_fmt = with_currency(pnl_value(row, "ProfitAndLoss/Total/ReportCcy"), report_currency)
    market_fmt = with_currency(pnl_value(row, "ProfitAndLoss/Total/Market/ReportCcy"), report_currency)
    other_fmt = with_currency(pnl_value(row, "ProfitAndLoss/Total/Other/ReportCcy"), report_currency)
    print(f"Total/ReportCcy {total_fmt} = Total/Market {market_fmt} + Total/Other {other_fmt}; Total/Fx {fx_fmt} is excluded")

---
## Section 10: Web UI Links

Open the portfolio and holdings directly in the LUSID Web UI.

In [ ]:
def to_iso_z(date):
    return date.isoformat(timespec="milliseconds").replace("+00:00", "Z")

sale_date_iso = to_iso_z(sale_date)
report_period_start_iso = to_iso_z(report_period_start_date)
report_period_end_iso = to_iso_z(report_period_end_date)

# Quotes
display(HTML(f'<a href="{api_url}/app/data-management/quotes?quotesScope={scope}" target="_blank">View Quotes</a>'))

# Holdings
display(HTML(f'<a href="{api_url}/app/dashboard/holdings?scope={scope}&code={portfolio_code}&entityType=Portfolio&effectiveDate={sale_date_iso}" target="_blank">View Holdings</a>'))

# Transactions
display(HTML(f'<a href="{api_url}/app/dashboard/transactions?scope={scope}&code={portfolio_code}&entityType=Portfolio&taxLots=false&withOrders=false&withAllocations=false&dashboardType=output&queryMode=TradeDate&showNonActiveTransactions=false&includeSettlementStatus=false" target="_blank">View Transactions</a>'))

# Instrument events - have to manually navigate back to December 2024
display(HTML(f'<a href="{api_url}/app/dashboard/instrument-events?scope={scope}&code={portfolio_code}&entityType=Portfolio&recipeScope={scope}&recipeCode={recipe_code}" target="_blank">View Instrument Events</a>'))

# Valuations
display(HTML(f'<a href="{api_url}/app/dashboard/valuations?scope={scope}&code={portfolio_code}&entityType=Portfolio&recipeScope={scope}&recipeCode={recipe_code}&effectiveDate={sale_date_iso}" target="_blank">View Valuations Dashboard</a>'))

# A2B
display(HTML(f'<a href="{api_url}/app/dashboard/a2b?scope={scope}&code={portfolio_code}&entityType=Portfolio&recipeScope={scope}&recipeCode={recipe_code}&effectiveDate=2024-12-31T23:59:59.999999Z&dateFrom={report_period_start_iso}&dateTo={report_period_end_iso}" target="_blank">View A2B Dashboard</a>'))


---
## Section 11: Luminesce queries

In [ ]:
from IPython.core.magic import (register_line_cell_magic)
import finbourne.sdk.services.luminesce as lum

luminesce_api = api_factory.build(lum.SqlExecutionApi)

@register_line_cell_magic
def luminesce(line, cell=None):
    query = cell if cell is not None else line

    # Get the IPython namespace to access user variables
    ipython = get_ipython()
    if ipython:
        # Get user namespace
        user_ns = ipython.user_ns

        # Format the query string with variables from the namespace
        try:
            # This will replace {variable_name} with the value of variable_name
            query = query.format(**user_ns)
        except KeyError as e:
            raise ValueError(f"Variable {e} not found in namespace. Make sure it's defined before using in the query.")
        except Exception as e:
            raise ValueError(f"Error formatting query: {e}")

    result_csv = luminesce_api.put_by_query_csv(body=query)

    return pd.read_csv(io.StringIO(result_csv))

# In an interactive session, we need to delete to avoid name conflicts for automagic to work on line magics.
del luminesce

### Standard A2B

In [ ]:
%%luminesce

select
      LusidInstrumentId as [Instrument Id],                                       
      InternalId,                                                                 
      sum(StartMarketValue) as StartMarketValue,
      sum(EndMarketValue) as EndMarketValue,                                      
      sum(CapGain) as [Sum of CapGain],                     
      sum(Carry) as [Sum of Carry],                                               
      sum(Flows) as [Sum of Flows]                                                
  from Lusid.Portfolio.AtoB
  where Recipe = '{scope}/{recipe_code}'
    and PortfolioScope = '{scope}'
    and PortfolioCode = '{portfolio_code}'
    and FromDate = #2024-10-19#
    and ToDate = #2025-01-01#
  group by LusidInstrumentId, InternalId 

### Splitting Holding vs Trading P&L

In [ ]:
%%luminesce

select
      LusidInstrumentId as [Instrument Id],                                       
      InternalId,                                                                 
      sum(StartMarketValue) as StartMarketValue,
      sum(EndMarketValue) as EndMarketValue,                                      
      sum(CapGain) as [Sum of CapGain],                     
      sum(Carry) as [Sum of Carry],                                               
      sum(Flows) as [Sum of Flows]                                                
  from Lusid.Portfolio.AtoB
  where Recipe = '{scope}/{recipe_code}'
    and PortfolioScope = '{scope}'
    and PortfolioCode = '{portfolio_code}'
    and FromDate = #2024-10-19#
    and ToDate = #2025-01-01#
    and TradingVsHolding = true
  group by LusidInstrumentId, InternalId 